In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
from datetime import datetime
import time as t

In [2]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36'
}

In [3]:
URL = input("Please Enter the URL of the product (Flipkart): ")
REVIEW_URL = URL.replace('/p/', '/product-reviews/')

In [4]:
try:
    response = requests.get(f'{REVIEW_URL}&page={1}', headers=headers)
    response.raise_for_status()
except requests.exceptions.RequestException as e:
    print(f"Error fetching the URL: {e}")
    exit()

html_content = response.content
soup = BeautifulSoup(html_content, "html.parser")

In [5]:
noOfReviews = soup.find('div', class_='css-146c3p1 r-1h7g6bg r-1enofrn r-1777fci r-14yzgew r-q4m81j').text
match = re.search(r'([\d,]+)\s+reviews', noOfReviews)
reviews_count = 0
if match:
    reviews_count = int(match.group(1).replace(',', ''))

noOfPages = reviews_count//10

In [ ]:
reviews = []

for i in range(1, noOfPages + 1):
    try:
        response = requests.get(f'{REVIEW_URL}&page={i}', headers=headers)

        # Basic rate limit handling
        if response.status_code == 429:
            print("Rate limited... sleeping")
            t.sleep(10)
            continue

        response.raise_for_status()

    except requests.exceptions.RequestException as e:
        print(f"Error fetching the URL: {e}")
        break

    html_content = response.content
    soup = BeautifulSoup(html_content, "html.parser")

    days_list = []
    content_list = []

    divs = soup.find_all('div', attrs={"dir": "auto"})

    for item in divs:
        text = item.get_text(strip=True)

        # Extract days
        if "ago" in text or re.search(r'20\d{2}', text):
            if re.search(r'20\d{2}', text):
                match = re.search(r"[A-Za-z]{3},\s*\d{4}", text)
                if match:
                    past_date = datetime.strptime(match.group(), "%b, %Y")
                    now = datetime.now()
                    days = (now - past_date).days
                    days_list.append(days)
            else:
                num = re.search(r"\d+", text)
                if num:
                    value = int(num.group())

                    if "today" in text:
                        days_list.append(0)
                    elif "day" in text:
                        days_list.append(value)
                    elif "month" in text:
                        days_list.append(value * 30)
                    elif "year" in text:
                        days_list.append(value * 365)

        # Extract content
        else:
            content = item.find('span', class_='css-1jxf684')
            if content:
                content_list.append(content.get_text(strip=True))

    day_index = 0

    for content in content_list:
        if day_index < len(days_list):
            day_value = days_list[day_index]
            day_index += 1
        else:
            day_value = days_list[-1] if days_list else None

        reviews.append({
            "days": day_value,
            "content": content
        })

    print(f"Page {i} done")
    t.sleep(2)

print(reviews)

Page 1 done
Page 2 done
Page 3 done
Page 4 done
Page 5 done
Page 6 done
Page 7 done
Page 8 done
Page 9 done
Page 10 done
Page 11 done
Page 12 done
Page 13 done
Page 14 done
Page 15 done
Page 16 done
Page 17 done
Page 18 done
Page 19 done
Page 20 done
Page 21 done
Page 22 done
Page 23 done
Page 24 done
Page 25 done
Page 26 done
Page 27 done
Page 28 done
Page 29 done
Page 30 done
Page 31 done
Page 32 done
Page 33 done
Page 34 done
Page 35 done
Page 36 done
Page 37 done
Page 38 done
Page 39 done
Page 40 done
Page 41 done
Page 42 done
Page 43 done
Page 44 done
Page 45 done
Page 46 done
Page 47 done
Page 48 done
Page 49 done
Page 50 done
Page 51 done
Page 52 done
Page 53 done
Page 54 done
Page 55 done
Page 56 done
Page 57 done
Page 58 done
Page 59 done
Page 60 done
Page 61 done
Page 62 done
Page 63 done
Page 64 done
Page 65 done
Page 66 done
Page 67 done
Page 68 done
Page 69 done
Page 70 done
Page 71 done
Page 72 done
Page 73 done
Page 74 done
Page 75 done
Page 76 done
Page 77 done
Page 78 

In [10]:
len(reviews)

1745

In [11]:
df = pd.DataFrame(reviews)
df.to_csv("product_reviews.csv", index=False)